# 🎬 LongCat-Avatar-1.5 Studio Master (Google Colab Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azadhossainofficial/LongCat-Avatar-1.5/blob/main/LongCat_Avatar_1_5_Colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/azadhossainofficial/LongCat-Avatar-1.5)

Welcome to **LongCat-Avatar-1.5 Studio Master** on Google Colab! This notebook deploys your complete conversational video avatar generation environment with:
- **Anchor Seamless Mode:** Flawless multi-segment continuity with Hann S-Curve blending
- **Natural Presenter Dynamics:** 100% natural organic upper-body posture, subtle chest breathing & micro-motions
- **Studio Master Dark UI:** 4-Slot sequential batch queue, real-time segment monitors & interactive video gallery
- **Free Public Tunnel:** Instant Cloudflare Tunnel access to your web UI with zero account needed

> ⚠️ **GPU Requirement:** Please ensure you are running on an **A100**, **L4**, or **V100** GPU (`Runtime` -> `Change runtime type` -> select `A100 GPU` or `L4 GPU`).

## ⚡ Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi

## 📦 Step 2: Clone Official Repository & Install Dependencies

In [ ]:
import os
%cd /content
if not os.path.exists('/content/LongCat-Avatar-1.5'):
    !git clone https://github.com/azadhossainofficial/LongCat-Avatar-1.5.git
%cd /content/LongCat-Avatar-1.5
!git pull

print('\n📦 Installing System Packages (FFmpeg, Libsndfile, Git-LFS)...')
!apt-get update -y -qq && apt-get install -y -qq ffmpeg libsndfile1 git-lfs procps net-tools > /dev/null

print('\n🐍 Installing Python Requirements...')
!pip install --upgrade pip -qq
!pip install huggingface_hub -qq
!pip install -r requirements.txt -qq
!pip install -r requirements_avatar.txt -qq
!pip install -r requirements_ui.txt -qq

print('\n✅ Environment setup completed successfully!')

## ⬇️ Step 3: Download Model Weights from HuggingFace

Downloads official LongCat-Video-Avatar-1.5 and Wan-VAE weights directly to `./weights/`.

In [ ]:
%cd /content/LongCat-Avatar-1.5
import os
os.environ['WEIGHTS_DIR'] = '/content/LongCat-Avatar-1.5/weights'
!python download_avatar_models.py
print('\n✅ All Model Weights Verified & Ready!')

## 🚀 Step 4: Launch Studio Master Server & Open Web UI

This cell launches the backend `server.py` on port `20100` and creates a **free, public, secure Cloudflare Tunnel** (`https://xxxx.trycloudflare.com`). Click the generated link to open the full LongCat Avatar 1.5 Studio UI in your browser!

In [ ]:
%cd /content/LongCat-Avatar-1.5
import subprocess
import time
import re

# 1. Install Cloudflared Tunnel Client
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Launch Studio Master Server on Port 20100
print('🎬 Starting LongCat-Avatar-1.5 Studio Master Server on port 20100...')
server_proc = subprocess.Popen(['python3', 'server.py', '20100'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(3)

# 3. Create Cloudflare Tunnel to port 20100
print('🌐 Opening Free Cloudflare Public Tunnel...')
cf_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:20100'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for _ in range(40):
    line = cf_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print('\n' + '='*72)
    print('🎉 LongCat-Avatar-1.5 Studio Master is LIVE on Google Colab!')
    print(f'👉 Open Web UI: {tunnel_url}')
    print('='*72 + '\n')
else:
    print('⚠️ Tunnel took longer to establish. Check cloudflared output below:')

# Stream server logs in real time
while True:
    line = server_proc.stdout.readline()
    if line:
        print(line.decode('utf-8', errors='ignore'), end='')
